# Stock Market Anomaly Detection
### CS 210: Data Management for Data Science
**Author:** Purabh Singh

End-to-end pipeline detecting anomalous trading days using Z-Score, Isolation Forest, and LOF.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from src.data_ingestion import DataIngestion, DEFAULT_TICKERS, DEFAULT_START, DEFAULT_END
from src.feature_engineering import FeatureEngineer
from src.anomaly_detection import AnomalyDetector
from src.database import StockDatabase
from src.visualization import Visualization
from src.fomc_events import get_fomc_dates, label_fomc_anomalies
from src.contagion_analysis import cross_stock_correlation, detect_sector_contagion
print('Setup OK. Tickers:', DEFAULT_TICKERS)

## 1. Data Loading & EDA
yfinance provides free OHLCV data. In the CS 210 relational model, each ticker has a one-to-many relationship with its price records stored in the `price_data` table.

In [ ]:
ingestion = DataIngestion()
price_data = {}
for ticker in ['AAPL', 'MSFT', 'TSLA']:
    df = ingestion.fetch_stock_data_by_date(ticker, start_date='2020-01-01', end_date='2024-12-31')
    if df is not None and not df.empty:
        price_data[ticker] = df
        print(f'{ticker}: {len(df)} days loaded')
price_data.get('AAPL', pd.DataFrame()).head()

In [ ]:
fig, axes = plt.subplots(len(price_data), 1, figsize=(14, 4*len(price_data)))
if len(price_data)==1: axes=[axes]
for ax, (ticker, df) in zip(axes, price_data.items()):
    col = 'close' if 'close' in df.columns else 'Close'
    ax.plot(df.index, df[col], lw=1, color='steelblue')
    ax.set_title(f'{ticker} Closing Price')
    ax.set_ylabel('Price ($)')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(price_data), figsize=(14, 4))
if len(price_data)==1: axes=[axes]
for ax, (ticker, df) in zip(axes, price_data.items()):
    col = 'close' if 'close' in df.columns else 'Close'
    r = df[col].pct_change().dropna()
    ax.hist(r, bins=80, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(r.mean(), color='red', ls='--', label=f'mean={r.mean():.4f}')
    ax.set_title(f'{ticker} Daily Returns')
    ax.legend()
plt.tight_layout(); plt.show()

## 2. Feature Engineering
50+ indicators computed from raw OHLCV. This is the data cleaning/transformation stage from CS 210 — converting raw relational data into an ML-ready feature matrix.

In [ ]:
engineer = FeatureEngineer()
features_dict = {}
for ticker, df in price_data.items():
    feat = engineer.engineer_features(df)
    features_dict[ticker] = feat
    print(f'{ticker}: {feat.shape[1]} features')

ticker = list(features_dict.keys())[0]
key_feats = [f for f in ['log_return','ma_20','volatility_20','rsi_14','volume_ratio'] if f in features_dict[ticker].columns]
if key_feats:
    plt.figure(figsize=(8,6))
    sns.heatmap(features_dict[ticker][key_feats].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title(f'{ticker} Feature Correlation')
    plt.tight_layout(); plt.show()

## 3. Anomaly Detection
Three methods applied. **Agreement score** (0-3) = number of methods flagging a day. Days with score ≥2 are high-confidence anomalies.

In [ ]:
detector = AnomalyDetector()
results_dict = {}
for ticker, feat in features_dict.items():
    res = detector.detect_anomalies(feat)
    results_dict[ticker] = res
    n = int(res['agreement_score'].ge(2).sum()) if 'agreement_score' in res.columns else 0
    print(f'{ticker}: {n} high-confidence anomalies')

t = list(results_dict.keys())[0]
results_dict[t][results_dict[t].get('agreement_score', pd.Series(0,index=results_dict[t].index))>=2][['z_score_flag','isolation_forest_flag','lof_flag','agreement_score']].head(10)

In [ ]:
ticker = list(results_dict.keys())[0]
feat = features_dict[ticker]
res = results_dict[ticker]
col = 'close' if 'close' in feat.columns else feat.columns[3]
fig = go.Figure()
fig.add_trace(go.Scatter(x=feat.index, y=feat[col], mode='lines', name='Close', line=dict(color='steelblue', width=1)))
if 'agreement_score' in res.columns:
    anom = res[res['agreement_score']>=2]
    if len(anom)>0:
        fig.add_trace(go.Scatter(x=anom.index, y=feat.loc[feat.index.isin(anom.index), col], mode='markers', name='Anomaly', marker=dict(color='red',size=8,symbol='x')))
fig.update_layout(title=f'{ticker} Price with Anomalies', xaxis_title='Date', yaxis_title='Price ($)', height=400)
fig.show()

## 4. FOMC Alignment Analysis
We test whether anomalies cluster around Federal Reserve FOMC meeting dates — our primary macroeconomic event hypothesis.

In [ ]:
import datetime
fomc_dates = get_fomc_dates()
print(f'FOMC dates 2015-2024: {len(fomc_dates)}')

near, far = 0, 0
for ticker, res in results_dict.items():
    if 'agreement_score' not in res.columns: continue
    for d in res[res['agreement_score']>=2].index:
        dd = pd.to_datetime(d).date()
        if any(abs((dd-f).days)<=2 for f in fomc_dates): near+=1
        else: far+=1

plt.figure(figsize=(7,5))
plt.bar(['Near FOMC (<=2d)', 'Far from FOMC (>2d)'], [near, far], color=['#e74c3c','#3498db'])
plt.title('Anomalies: FOMC Proximity')
plt.ylabel('Count')
plt.tight_layout(); plt.show()
print(f'Near FOMC: {near}, Far: {far}, FOMC explanation rate: {near/(near+far):.1%}' if near+far>0 else 'No anomalies')

In [ ]:
# Cross-stock correlation heatmap
if len(price_data)>=2:
    corr_df = cross_stock_correlation(price_data)
    if not corr_df.empty:
        try:
            last_date = corr_df.index.get_level_values(0)[-1]
            snap = corr_df.loc[last_date]
            plt.figure(figsize=(6,5))
            sns.heatmap(snap, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
            plt.title('20-Day Rolling Return Correlation')
            plt.tight_layout(); plt.show()
        except Exception as e:
            print('Skipped:', e)

## 5. Methodology & CS 210 Connections

### Relational Database Design
Four normalized SQLite tables: `stocks`, `price_data`, `news_articles`, `anomalies`. This prevents data redundancy and supports efficient JOIN queries (e.g., matching anomaly dates with news sentiment).

### Data Cleaning
Raw yfinance data is cleaned via: forward-fill of missing days, split/dividend adjustment, NaN removal from rolling window calculations, and column standardization.

### Unsupervised ML
No labeled anomaly ground truth exists. The three methods are complementary:
- **Z-Score**: statistical deviation from rolling mean
- **Isolation Forest**: random partitioning — fewer splits = more isolated = more anomalous
- **LOF**: local density — anomalies have much lower density than their k nearest neighbors

The agreement-score ensemble (flag only when ≥2 methods agree) reduces false positives.

## 6. Limitations & Future Work

1. **NewsAPI free tier** — only last 30 days; historical news coverage incomplete
2. **Survivorship bias** — only currently-listed stocks analyzed
3. **Contamination hyperparameter** — IsolationForest 10% assumption affects sensitivity
4. **No ground-truth labels** — precision-by-explanation is a proxy, not true precision
5. **Future work** — incorporate SEC EDGAR filings, earnings transcripts, or social media sentiment for richer explanation